# Итоговый Spark-проект eBay — 30 заданий

Практика на eBay; решений нет.

## Результаты обучения

После **Итоговый Spark-проект** вы должны объяснить transformation как plan, предсказать action/jobs/shuffle, связать schema/grain с результатом и доказать физическую эффективность через explain/UI/metrics.

## Ментальная модель исполнения

Data product объединяет grain/schema, layers, DQ, physical design, plan, incremental strategy и runbook. Код без доказательств идемпотентности и reconciliation не закончен.

```text
transformations → unresolved logical plan
       ↓ analysis (catalog/types)
   optimized logical plan (Catalyst)
       ↓ physical planning / AQE
job → stage → shuffle → stage
       tasks             tasks
       └──── executors ─────┘
```
Action создаёт job. Один notebook/application может породить много jobs, а один job —
несколько stages. `repartition`, join и groupBy часто добавляют Exchange.

## Данные eBay

Grain eBay — `itemid` в `snapshot_dt`; 2 501 511 строк, 24 колонки, Parquet/Snappy.
Partition column — дата снимка. Цена, продавец, категории и доставка денормализованы.
Перед `latest item` или dedup проверяйте уникальность пары и задавайте tie-breaker.

Полная схема и проверки качества находятся в `data-catalog`. Raw read-only, результаты — в личном `spark_training`.

## Алгоритм решения

1. Зафиксируйте входной и целевой grain. 2. Выберите только нужные columns/rows. 3. Соберите transformation без action. 4. Проверьте schema и explain. 5. Предскажите partitions/shuffle. 6. Выполните минимальный action/write. 7. Повторно прочитайте и сверяйте keys/metrics. 8. Сохраните evidence.

Принимайте каждый слой отдельно и сохраняйте explain/metrics/files вместе с бизнес-проверками.

## Типичные ошибки

- Вызывать count/show после каждого шага и создавать лишние jobs.
- Использовать Python UDF при наличии встроенной функции.
- Делать repartition без понимания Exchange и целевого файла.
- Broadcast большой стороны или collect на driver.
- Кэшировать одноразовый DataFrame без materialization/unpersist.
- Измерять скорость при разных результатах или непрогретом JVM.

## Самопроверка

1. Какой action создаёт job? 2. Где появится shuffle? 3. Сколько input/output partitions? 4. Видит ли Catalyst выражение? 5. Каков grain после JOIN/window? 6. Как проверить idempotent rerun?

## Подробная теория

### 1. Проектирование

До кода задайте grain, SLA, ключи, DQ, обновление и владельца.

### 2. Слои

raw→typed→accepted/reject→dimensions/fact→marts, каждый переход сверяется.

### 3. Физика

Partition, files, compression, cache и join strategy подтверждаются данными и plan.

### 4. Эксплуатация

Нужны run_id, rerun дня, late data, cleanup и мониторинг.

### 5. Приёмка

Докажите schema, uniqueness, reconciliation, pruning, plan и идемпотентность.

## Сдача

Каждое задание записывает непустой Parquet в личный HDFS и evidence с transformation, observation и explanation. Checker использует активную SparkSession.

In [ ]:
import os,sys
sys.path.insert(0,'/opt/lab/spark-training')
from check_task import check_task,save_evidence
from pyspark.sql import SparkSession,functions as F,types as T,Window
spark=SparkSession.builder.appName('spark-training').enableHiveSupport().getOrCreate()
USER=os.environ.get('HDFS_USER',os.environ.get('HADOOP_USER_NAME','student'))
ROOT=f'hdfs://namenode:8020/user/{USER}/spark_training'
SOURCE='hdfs://namenode:8020/data/raw/ebay'
ebay=spark.read.parquet(SOURCE)
print('Spark',spark.version,'rows',ebay.count(),'columns',len(ebay.columns))

### Задание 1. source inventory

Создайте результат по теме **source inventory** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_01")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',1)

### Задание 2. source schema

Создайте результат по теме **source schema** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_02")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',2)

### Задание 3. source profile

Создайте результат по теме **source profile** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_03")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',3)

### Задание 4. grain

Создайте результат по теме **grain** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_04")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',4)

### Задание 5. business key

Создайте результат по теме **business key** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_05")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',5)

### Задание 6. null profile

Создайте результат по теме **null profile** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_06")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',6)

### Задание 7. duplicate profile

Создайте результат по теме **duplicate profile** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_07")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',7)

### Задание 8. typed staging

Создайте результат по теме **typed staging** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_08")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',8)

### Задание 9. reject layer

Создайте результат по теме **reject layer** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_09")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',9)

### Задание 10. accepted layer

Создайте результат по теме **accepted layer** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_10")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',10)

### Задание 11. dedup current

Создайте результат по теме **dedup current** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_11")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',11)

### Задание 12. price normalization

Создайте результат по теме **price normalization** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_12")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',12)

### Задание 13. shipping normalization

Создайте результат по теме **shipping normalization** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_13")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',13)

### Задание 14. seller dimension

Создайте результат по теме **seller dimension** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_14")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',14)

### Задание 15. category dimension

Создайте результат по теме **category dimension** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_15")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',15)

### Задание 16. listing fact

Создайте результат по теме **listing fact** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_16")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',16)

### Задание 17. daily snapshot

Создайте результат по теме **daily snapshot** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_17")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',17)

### Задание 18. price statistics

Создайте результат по теме **price statistics** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_18")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',18)

### Задание 19. seller metrics

Создайте результат по теме **seller metrics** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_19")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',19)

### Задание 20. category metrics

Создайте результат по теме **category metrics** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_20")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',20)

### Задание 21. quality mart

Создайте результат по теме **quality mart** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_21")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',21)

### Задание 22. partition strategy

Создайте результат по теме **partition strategy** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_22")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',22)

### Задание 23. file sizing

Создайте результат по теме **file sizing** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_23")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',23)

### Задание 24. cache decision

Создайте результат по теме **cache decision** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_24")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',24)

### Задание 25. join strategy

Создайте результат по теме **join strategy** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_25")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',25)

### Задание 26. plan evidence

Создайте результат по теме **plan evidence** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_26")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',26)

### Задание 27. incremental day

Создайте результат по теме **incremental day** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_27")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',27)

### Задание 28. idempotent rerun

Создайте результат по теме **idempotent rerun** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_28")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',28)

### Задание 29. reconciliation

Создайте результат по теме **reconciliation** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_29")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',29)

### Задание 30. final acceptance

Создайте результат по теме **final acceptance** и запишите `mode("overwrite").parquet(f"{ROOT}/capstone/task_30")`. До action предскажите plan/число строк; после проверьте schema, ключи, NULL и partitions. Сохраните evidence.

<details><summary>Подсказка</summary>

Начните с минимального `ebay.select(...)`, вызовите `explain`, затем проверьте повторным чтением.

</details>

<!-- task-card-spark-v1 -->
#### Карточка выполнения

- **Учебная цель:** какой Spark-механизм должен стать наблюдаемым?
- **Контракт данных:** входной/целевой grain, key, schema, NULL и ожидаемый объём.
- **Логический план:** projection, filters, joins, aggregates/windows до action.
- **Физический прогноз:** partitions, Exchange, sort, broadcast, jobs/stages.
- **Проверка:** `printSchema`, `explain("formatted")`, keys/metrics и повторное чтение Parquet.
- **Эксплуатация:** write mode, file count, повтор запуска, cleanup/unpersist.
- **Evidence/checker:** объясните наблюдаемый plan и результат, а не просто вызванный API.

Подсказка не определяет готовую цепочку transformations: её нужно вывести из grain и контракта.

In [ ]:
# Ваше решение
# result = ...
# result.write.mode("overwrite").parquet(...)
# save_evidence(spark,...)

In [ ]:
check_task(spark,'capstone',30)